# 03 — Feature Engineering

**Project**: LLM-Based Startup Risk Profiling and Funding Outcome Prediction

This notebook covers:
1. Financial ratio features
2. Temporal features
3. Text-derived features (readability, risk keywords)
4. LLM embeddings (optional)
5. Pipeline construction
6. Data leakage prevention

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import (
    FeatureEngineer, TextFeatureExtractor, RiskKeywordExtractor,
    build_preprocessing_pipeline
)

pd.set_option('display.max_columns', None)

In [ ]:
df = pd.read_csv('../data/startup_cleaned.csv')
print(f'Shape: {df.shape}')

## 1. Financial Features

In [ ]:
fe = FeatureEngineer()
df_eng = fe.transform(df)

engineered = ['company_age', 'funding_velocity', 'rounds_per_year',
              'time_to_first_funding', 'funding_span', 'avg_round_size',
              'description_length', 'description_readability']

df_eng[engineered].describe()

## 2. Risk Keyword Features

In [ ]:
rke = RiskKeywordExtractor()
risk_features = rke.transform(df)
risk_features.head(10)

## 3. Risk Keywords vs Outcome

In [ ]:
risk_df = risk_features.copy()
risk_df['funding_success'] = df['funding_success']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, ['high_risk_score', 'medium_risk_score', 'low_risk_score']):
    risk_df.boxplot(column=col, by='funding_success', ax=ax)
    ax.set_xticklabels(['Failure', 'Success'])
    ax.set_title(col)
    plt.suptitle('')

plt.tight_layout()
plt.savefig('../reports/figures/risk_keywords_by_outcome.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Build Full Pipeline

In [ ]:
pipe = build_preprocessing_pipeline(use_llm_embeddings=False)
print('Pipeline structure:')
print(pipe)

pipe.fit(df.drop(columns=['funding_success']))
print('\nPipeline fitted successfully.')

## 5. Data Leakage Prevention

Key measures:
- Pipeline fit on training data only
- No target-derived features
- All features available at prediction time
- SimpleImputer and StandardScaler fit on train, applied to val/test